# Waypoint final modelling notebook

Backend-owned, offline workflow. Authorized data/artifacts remain in `backend/private/intelligence`; do not publish executed outputs or model derivatives. See README.md for training cutoffs, candidate limitations and exact dependency versions. This compact Task 1 candidate replaces missing source artifacts; it is not the source README’s boosted winner.


In [ ]:
from pathlib import Path
import sys
import json

project = next(
    root
    for root in [Path.cwd(), *Path.cwd().parents]
    if (root / "backend/intelligence/features.py").is_file()
)
for relative in [
    "backend/intelligence",
    "backend/services/intelligence/src",
    "backend/packages/contracts",
]:
    sys.path.insert(0, str(project / relative))

In [ ]:
from runtime import ROOT
from features import prepare_task1
from waypoint_intelligence import engine

## Labels and planned features

Service minutes = leave − max(actual arrival, window opening); late = actual arrival > closing. Labels are targets only. Planned route legs join on route and sequence. Outlet/vehicle/calendar/traffic/road/handling references enrich planned features. Prefix loads and total route size require complete planned routes. No post-delivery labels are accepted by the inference API.


In [ ]:
labelled = prepare_task1()
train = labelled[labelled.dispatch_date < "2026-01-05"]
holdout = labelled[labelled.dispatch_date.between("2026-01-05", "2026-02-14")]
assert len(train) and len(holdout)
assert train.dispatch_date.max() < holdout.dispatch_date.min()
print(
    {
        "train_rows": len(train),
        "holdout_rows": len(holdout),
        "label_ranges": {
            "service_min": [
                float(labelled.service.min()),
                float(labelled.service.max()),
            ]
        },
    }
)

## Explicit fitting and evaluation

Set RUN_TRAINING to True to reproduce Task 1 fits. Forecast inference loads the supplied trained Ridge ensemble, never a proprietary API or pretrained foundation model. All holdout labels stay outside fitting. Task 2A counts every requested order, including deferred/not-run demand, and uses calendar ISO periods.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from train_runtime import main as train_runtime

    train_runtime()
print(json.loads((ROOT / "reports/runtime_evaluation.json").read_text()))
forecast = engine.demand_forecast()
print({"forecast_holdout": forecast.backtest, "model": forecast.model_version})

## Saved-model inference: inputs and predictions

This final cell loads verified saved models and demonstrates Task 1 and Task 2A inference. Full template exports and independent allocation validation are generated by export_runtime.py / validate_submissions.py; the notebook does not run a training grid.


In [ ]:
planned = engine.test_frame().head(3)
print("Task 1 planned inputs:")
print(
    planned[
        [
            "delivery_id",
            "brand",
            "order_weight_kg",
            "order_volume_m3",
            "planned_arrival_time",
        ]
    ].to_string(index=False)
)
print("Task 1 saved-model predictions:")
print(engine.predict_ids(planned.delivery_id.tolist()).model_dump_json(indent=2))
print("Task 2A inputs:")
print(engine.read("Test Data", "task2a_test_inputs.csv").head(3).to_string(index=False))
print("Task 2A saved-model predictions:")
print(json.dumps([row.model_dump() for row in forecast.rows[:3]], indent=2))